In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import shap
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from datetime import datetime
import sys
sys.path.append(str(Path('..').resolve()))
from config import (DATA_PROCESSED, MODELS_DIR, SCALER_PATH,
                    FEATURE_COLS_PATH, RANDOM_STATE)

from sklearn.metrics import f1_score, r2_score, mean_absolute_error
import numpy as np

print("Imports OK")

Imports OK


In [2]:
# Load everything the API will need
classifier  = joblib.load(MODELS_DIR / "classifier_best.pkl")
regressor   = joblib.load(MODELS_DIR / "regressor_best.pkl")
scaler      = joblib.load(MODELS_DIR / "scaler.pkl")

with open(FEATURE_COLS_PATH) as f:
    feature_columns = json.load(f)

with open(MODELS_DIR / "classifier_threshold.json") as f:
    thresh_data = json.load(f)

with open(MODELS_DIR / "regressor_meta.json") as f:
    reg_meta = json.load(f)

clf_threshold       = thresh_data["threshold"]
reg_feature_columns = [c for c in feature_columns if c != 'cost_per_hour']
cost_idx            = feature_columns.index('cost_per_hour')

# Build SHAP explainers once at load time (expensive to rebuild per request)
clf_explainer = shap.TreeExplainer(classifier)
reg_explainer = shap.TreeExplainer(regressor)

print("All artifacts loaded:")
print(f"  Classifier  : {thresh_data['model']}")
print(f"  Regressor   : {reg_meta['model']}")
print(f"  Threshold   : {clf_threshold}")
print(f"  Features    : {len(feature_columns)}")
print(f"  SHAP ready  : ✓")

All artifacts loaded:
  Classifier  : XGBoost (threshold tuned)
  Regressor   : XGBoost
  Threshold   : 0.5500000000000002
  Features    : 95
  SHAP ready  : ✓


In [3]:
def predict_single(raw_row: dict) -> dict:
    """
    Full stacked pipeline prediction for one resource reading.

    Input  : raw_row — dict with all feature values (post feature engineering)
    Output : dict with anomaly flag, probability, predicted cost, SHAP reasons

    This is exactly what src/predict.py will expose to the API.
    """

    # ── Step 1: Build feature dataframe ──────────────────────────────
    row_df = pd.DataFrame([raw_row])

    # Ensure all feature columns exist — fill missing with 0
    for col in feature_columns:
        if col not in row_df.columns:
            row_df[col] = 0.0

    row_df = row_df[feature_columns]

    # ── Step 2: Scale ─────────────────────────────────────────────────
    row_scaled_full = scaler.transform(row_df)          # shape: (1, 95)
    row_scaled_reg  = np.delete(row_scaled_full,         # shape: (1, 94)
                                cost_idx, axis=1)

    # ── Step 3: Stage 1 — classify ────────────────────────────────────
    anomaly_proba   = float(classifier.predict_proba(row_scaled_full)[0, 1])
    is_anomaly      = int(anomaly_proba >= clf_threshold)

    # ── Step 4: Stage 2 — regress (stacked) ───────────────────────────
    row_stacked     = np.column_stack([row_scaled_reg,
                                       [[anomaly_proba]]])  # shape: (1, 95)
    predicted_cost  = float(regressor.predict(row_stacked)[0])

    # ── Step 5: SHAP explanation ──────────────────────────────────────
    # Classifier SHAP — top 5 features driving anomaly decision
    clf_shap_vals   = clf_explainer.shap_values(row_scaled_full)

    if isinstance(clf_shap_vals, list):
        clf_sv = clf_shap_vals[1][0]
    elif len(np.array(clf_shap_vals).shape) == 3:
        clf_sv = clf_shap_vals[0, :, 1]
    else:
        clf_sv = clf_shap_vals[0]

    # Regressor SHAP — top 5 features driving cost prediction
    reg_shap_vals   = reg_explainer.shap_values(row_stacked)

    if isinstance(reg_shap_vals, list):
        reg_sv = reg_shap_vals[0][0]
    elif len(np.array(reg_shap_vals).shape) == 3:
        reg_sv = reg_shap_vals[0, :, 0]
    else:
        reg_sv = reg_shap_vals[0]

    # Build top-5 SHAP reason lists
    clf_top5_idx    = np.argsort(np.abs(clf_sv))[-5:][::-1]
    reg_feat_names  = reg_feature_columns + ["anomaly_probability"]
    reg_top5_idx    = np.argsort(np.abs(reg_sv))[-5:][::-1]

    clf_reasons = [
        {
            "feature" : feature_columns[i],
            "value"   : float(row_df.iloc[0][feature_columns[i]]),
            "shap"    : round(float(clf_sv[i]), 4),
            "impact"  : "↑ anomaly" if clf_sv[i] > 0 else "↓ anomaly"
        }
        for i in clf_top5_idx
    ]

    reg_reasons = [
        {
            "feature" : reg_feat_names[i] if i < len(reg_feat_names) else f"feat_{i}",
            "value"   : float(row_stacked[0][i]),
            "shap"    : round(float(reg_sv[i]), 4),
            "impact"  : "↑ cost" if reg_sv[i] > 0 else "↓ cost"
        }
        for i in reg_top5_idx
    ]

    # ── Step 6: Severity label ────────────────────────────────────────
    if not is_anomaly:
        severity = "normal"
    elif anomaly_proba >= 0.90:
        severity = "critical"
    elif anomaly_proba >= 0.75:
        severity = "high"
    else:
        severity = "medium"

    # ── Step 7: Assemble output ───────────────────────────────────────
    return {
        "timestamp"       : datetime.now().isoformat(),
        "resource_id"     : raw_row.get("resource_id", "unknown"),
        "resource_type"   : raw_row.get("resource_type", "unknown"),
        "is_anomaly"      : is_anomaly,
        "anomaly_prob"    : round(anomaly_proba, 4),
        "severity"        : severity,
        "predicted_cost"  : round(predicted_cost, 4),
        "clf_reasons"     : clf_reasons,
        "reg_reasons"     : reg_reasons,
    }

print("predict_single() defined ✓")

predict_single() defined ✓


In [4]:
# Load test data to grab real rows
df = pd.read_csv(DATA_PROCESSED / "cloud_metrics_featured.csv")
n         = len(df)
val_end   = int(n * 0.85)
df_test   = df.iloc[val_end:].reset_index(drop=True)

# Find a clearly normal row
normal_rows = df_test[df_test['is_anomaly'] == 0]
normal_row  = normal_rows.iloc[10].to_dict()

print("Testing with a NORMAL row...")
print(f"  Actual label     : {int(normal_row['is_anomaly'])}")
print(f"  CPU utilization  : {normal_row['cpu_utilization']:.2f}%")
print(f"  Cost per hour    : ${normal_row['cost_per_hour']:.4f}")

result_normal = predict_single(normal_row)

print(f"\n── PREDICTION OUTPUT ──────────────────")
print(f"  is_anomaly     : {result_normal['is_anomaly']}")
print(f"  anomaly_prob   : {result_normal['anomaly_prob']}")
print(f"  severity       : {result_normal['severity']}")
print(f"  predicted_cost : ${result_normal['predicted_cost']:.4f}/hr")
print(f"\n── TOP 5 CLASSIFIER REASONS ───────────")
for r in result_normal['clf_reasons']:
    print(f"  {r['feature']:<35} shap={r['shap']:>8.4f}  {r['impact']}")
print(f"\n── TOP 5 REGRESSOR REASONS ────────────")
for r in result_normal['reg_reasons']:
    print(f"  {r['feature']:<35} shap={r['shap']:>8.4f}  {r['impact']}")

Testing with a NORMAL row...
  Actual label     : 0
  CPU utilization  : 63.69%
  Cost per hour    : $6.4933

── PREDICTION OUTPUT ──────────────────
  is_anomaly     : 0
  anomaly_prob   : 0.0001
  severity       : normal
  predicted_cost : $6.7231/hr

── TOP 5 CLASSIFIER REASONS ───────────
  hour_cos                            shap= -2.6353  ↓ anomaly
  cost_per_cpu                        shap= -2.5837  ↓ anomaly
  cost_per_hour_lag_24h               shap= -1.7651  ↓ anomaly
  day_of_week                         shap= -0.9058  ↓ anomaly
  memory_utilization                  shap= -0.7872  ↓ anomaly

── TOP 5 REGRESSOR REASONS ────────────
  cost_per_hour_roll_mean_6h          shap=  1.8347  ↑ cost
  cpu_utilization                     shap=  1.3390  ↑ cost
  resource_pressure                   shap=  0.3336  ↑ cost
  cost_per_hour_lag_24h               shap=  0.1934  ↑ cost
  cost_per_request                    shap=  0.1720  ↑ cost


In [5]:
# Find a clearly anomalous row
anomaly_rows = df_test[df_test['is_anomaly'] == 1]
anomaly_row  = anomaly_rows.iloc[5].to_dict()

print("Testing with an ANOMALY row...")
print(f"  Actual label     : {int(anomaly_row['is_anomaly'])}")
print(f"  CPU utilization  : {anomaly_row['cpu_utilization']:.2f}%")
print(f"  Cost per hour    : ${anomaly_row['cost_per_hour']:.4f}")

result_anomaly = predict_single(anomaly_row)

print(f"\n── PREDICTION OUTPUT ──────────────────")
print(f"  is_anomaly     : {result_anomaly['is_anomaly']}")
print(f"  anomaly_prob   : {result_anomaly['anomaly_prob']}")
print(f"  severity       : {result_anomaly['severity']}")
print(f"  predicted_cost : ${result_anomaly['predicted_cost']:.4f}/hr")
print(f"\n── TOP 5 CLASSIFIER REASONS ───────────")
for r in result_anomaly['clf_reasons']:
    print(f"  {r['feature']:<35} shap={r['shap']:>8.4f}  {r['impact']}")
print(f"\n── TOP 5 REGRESSOR REASONS ────────────")
for r in result_anomaly['reg_reasons']:
    print(f"  {r['feature']:<35} shap={r['shap']:>8.4f}  {r['impact']}")

Testing with an ANOMALY row...
  Actual label     : 1
  CPU utilization  : 21.20%
  Cost per hour    : $14.2616

── PREDICTION OUTPUT ──────────────────
  is_anomaly     : 1
  anomaly_prob   : 1.0
  severity       : critical
  predicted_cost : $13.3554/hr

── TOP 5 CLASSIFIER REASONS ───────────
  cost_per_hour_zscore                shap=  6.1262  ↑ anomaly
  cost_per_cpu                        shap=  2.8596  ↑ anomaly
  dow_cos                             shap= -2.3524  ↓ anomaly
  hour                                shap= -2.3466  ↓ anomaly
  cost_per_request                    shap=  1.1308  ↑ anomaly

── TOP 5 REGRESSOR REASONS ────────────
  cost_per_hour_zscore                shap=  4.3455  ↑ cost
  cost_per_hour_roll_mean_6h          shap=  3.2631  ↑ cost
  anomaly_probability                 shap=  1.7188  ↑ cost
  cost_per_cpu                        shap=  1.3596  ↑ cost
  cost_per_request                    shap=  0.8338  ↑ cost


In [7]:
print("Running batch prediction on full test set...")
print("(This validates the pipeline end-to-end on 3,240 rows)\n")

y_true_clf  = []
y_pred_clf  = []
y_true_reg  = []
y_pred_reg  = []

for idx, (i, row) in enumerate(df_test.iterrows()):
    raw = row.to_dict()
    out = predict_single(raw)

    y_true_clf.append(int(raw['is_anomaly']))
    y_pred_clf.append(out['is_anomaly'])
    y_true_reg.append(float(raw['cost_per_hour']))
    y_pred_reg.append(out['predicted_cost'])

    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1} / {len(df_test)} rows...")

y_true_clf = np.array(y_true_clf)
y_pred_clf = np.array(y_pred_clf)
y_true_reg = np.array(y_true_reg)
y_pred_reg = np.array(y_pred_reg)

clf_f1  = f1_score(y_true_clf, y_pred_clf)
reg_r2  = r2_score(y_true_reg, y_pred_reg)
reg_mae = mean_absolute_error(y_true_reg, y_pred_reg)

print(f"\n{'='*50}")
print(f"END-TO-END PIPELINE VALIDATION")
print(f"{'='*50}")
print(f"Test rows processed  : {len(df_test):,}")
print(f"Classifier F1        : {clf_f1:.4f}")
print(f"Regressor R²         : {reg_r2:.4f}")
print(f"Regressor MAE        : ${reg_mae:.4f}/hr")
print(f"\nExpected (from notebooks 04/05):")
print(f"  Classifier F1  ~ 0.9348")
print(f"  Regressor R²   ~ 0.9923")
print(f"  ✓ if within ±0.02 of notebook scores")

Running batch prediction on full test set...
(This validates the pipeline end-to-end on 3,240 rows)

  Processed 500 / 3240 rows...
  Processed 1000 / 3240 rows...
  Processed 1500 / 3240 rows...
  Processed 2000 / 3240 rows...
  Processed 2500 / 3240 rows...
  Processed 3000 / 3240 rows...

END-TO-END PIPELINE VALIDATION
Test rows processed  : 3,240
Classifier F1        : 0.9348
Regressor R²         : 0.9912
Regressor MAE        : $0.1171/hr

Expected (from notebooks 04/05):
  Classifier F1  ~ 0.9348
  Regressor R²   ~ 0.9923
  ✓ if within ±0.02 of notebook scores
